# Phase 5 — NB3: Batch Evaluation — 4 New Experiments

**Goal:** End-to-end joint evaluation of 4 new Stage 2 approaches:
1. Neighbor Perturbation flip=0.2
2. Neighbor Perturbation flip=0.4
3. Two-Head Additive lambda=0.2
4. Perturbation 0.2 + Two-Head

All use retrieval + aux_loss=0.1. Baselines (aux loss, no-ret) already evaluated.

**Target:** Joint F1 > 0.7696 (no-retrieval baseline)

**Input:**
- `duclm318/p5-nb1-stage1` — Stage 1 ckpt + processed data
- `duclm318/p5-nb2-stage2` — 4 new Stage 2 checkpoints
- `duclm318/p5-embed-v4` — embedding ckpt

## 0. Setup

In [ ]:
!pip install -q transformers faiss-cpu lxml scikit-learn pyyaml iterative-stratification

In [ ]:
import os, sys, json, shutil, subprocess, re

!git clone https://github.com/lucminhduc3108/Retrieval-ABSA.git /kaggle/working/repo
os.chdir('/kaggle/working/repo')
sys.path.insert(0, '/kaggle/working/repo')
print('Working dir:', os.getcwd())

In [ ]:
import torch, gc
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
gc.collect()
torch.cuda.empty_cache()

## 0b. Experiment Configuration

In [ ]:
STAGE1_VARIANT = 'cataware'
if STAGE1_VARIANT == 'asl':
    STAGE1_CONFIG = 'configs/stage1_2014.yaml'
    STAGE1_CKPT_NAME = 'stage1_2014_best.pt'
else:
    STAGE1_CONFIG = 'configs/stage1_2014_cataware.yaml'
    STAGE1_CKPT_NAME = 'stage1_2014_cataware_best.pt'

EXPERIMENTS = [
    {"name": "Perturb 0.2",       "ckpt": "stage2_2014_perturb02_best.pt",         "config": "configs/stage2_2014_perturb02.yaml",         "tag": "perturb02"},
    {"name": "Perturb 0.4",       "ckpt": "stage2_2014_perturb04_best.pt",         "config": "configs/stage2_2014_perturb04.yaml",         "tag": "perturb04"},
    {"name": "Two-Head",          "ckpt": "stage2_2014_twohead_best.pt",           "config": "configs/stage2_2014_twohead.yaml",           "tag": "twohead"},
    {"name": "Perturb02+TwoHead", "ckpt": "stage2_2014_perturb02_twohead_best.pt", "config": "configs/stage2_2014_perturb02_twohead.yaml", "tag": "perturb02_twohead"},
]

print(f'Stage 1: {STAGE1_VARIANT}')
print(f'Experiments: {len(EXPERIMENTS)}')
for exp in EXPERIMENTS:
    print(f'  - {exp["name"]}: {exp["ckpt"]} -> {exp["config"]}')

## 0c. Wire Data & Checkpoints

In [ ]:
def find_input(name):
    for p in [f'/kaggle/input/{name}',
              f'/kaggle/input/datasets/duclm318/{name}',
              f'/kaggle/input/datasets/lcminhc/{name}']:
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f'Dataset {name} not found')

NB1 = find_input('p5-nb1-stage1')
NB2 = find_input('p5-nb2-stage2')
EMB = find_input('p5-embed-v4')

print(f'NB1: {NB1}')
print(f'NB2: {NB2} -> {os.listdir(NB2)}')
print(f'EMB: {EMB}')

# Stage 1
os.makedirs('checkpoints/stage1', exist_ok=True)
shutil.copy(f'{NB1}/{STAGE1_CKPT_NAME}', 'checkpoints/stage1/best.pt')
print(f'\nStage 1 ckpt: {STAGE1_CKPT_NAME}')

# Embedding
os.makedirs('checkpoints/embedding_2014', exist_ok=True)
shutil.copy(f'{EMB}/embedding_v4_s2_best.pt', 'checkpoints/embedding_2014/best.pt')
print('Embedding ckpt wired.')

# Processed data
os.makedirs('data/processed', exist_ok=True)
shutil.copy(f'{NB1}/category_detection.jsonl', 'data/processed/category_detection.jsonl')
shutil.copy(f'{NB1}/sentiment_records.jsonl', 'data/processed/sentiment_records.jsonl')
print('Data files wired.')

# Stage 2 — wire all 4 checkpoints
wired = []
for exp in EXPERIMENTS:
    src = f'{NB2}/{exp["ckpt"]}'
    dst_dir = f'checkpoints/stage2_{exp["tag"]}'
    os.makedirs(dst_dir, exist_ok=True)
    if os.path.exists(src):
        shutil.copy(src, f'{dst_dir}/best.pt')
        wired.append(exp["name"])
        print(f'  {exp["ckpt"]}: {os.path.getsize(src)/1e6:.1f} MB')
    else:
        print(f'  WARNING: {exp["ckpt"]} NOT FOUND — will skip')

print(f'\nWired {len(wired)}/{len(EXPERIMENTS)} checkpoints: {wired}')

# Build FAISS index
os.makedirs('indexes', exist_ok=True)
!python scripts/03_build_index.py \
    --embedding_ckpt checkpoints/embedding_2014/best.pt \
    --input data/processed/sentiment_records.jsonl \
    --out_dir indexes/

## 1. Run Evaluations

In [ ]:
os.makedirs('logs', exist_ok=True)
results = []

for exp in EXPERIMENTS:
    ckpt_path = f'checkpoints/stage2_{exp["tag"]}/best.pt'
    if not os.path.exists(ckpt_path):
        print(f'\n=== SKIP {exp["name"]} — checkpoint missing ===')
        continue

    print(f'\n{"=" * 60}')
    print(f'Evaluating: {exp["name"]}')
    print(f'{"=" * 60}')

    gc.collect()
    torch.cuda.empty_cache()

    cmd = [
        'python', 'scripts/05_evaluate_joint.py',
        '--stage1_ckpt', 'checkpoints/stage1/best.pt',
        '--stage2_ckpt', ckpt_path,
        '--embedding_ckpt', 'checkpoints/embedding_2014/best.pt',
        '--index_dir', 'indexes/',
        '--stage1_config', STAGE1_CONFIG,
        '--stage2_config', exp['config'],
        '--retrieval_config', 'configs/retrieval_v2.yaml',
        '--pred_strategy', 'per_category',
    ]

    result = subprocess.run(cmd, capture_output=True, text=True)
    print(result.stdout[-2000:] if len(result.stdout) > 2000 else result.stdout)
    if result.returncode != 0:
        print(f'ERROR: {result.stderr[-500:]}')
        continue

    # Rename output log
    src_log = 'logs/joint_eval_retrieval.md'
    dst_log = f'logs/joint_eval_{exp["tag"]}.md'
    if os.path.exists(src_log):
        shutil.copy(src_log, dst_log)

    # Parse metrics from stdout
    metrics = {'name': exp['name']}
    for line in result.stdout.split('\n'):
        if 'Cat F1=' in line:
            m = re.search(r'Cat F1=([\d.]+).*Joint F1=([\d.]+).*Sent Acc\|CC=([\d.]+).*Sent MacF1\|CC=([\d.]+)', line)
            if m:
                metrics['cat_f1'] = float(m.group(1))
                metrics['joint_f1'] = float(m.group(2))
                metrics['sent_acc'] = float(m.group(3))
                metrics['sent_macf1'] = float(m.group(4))
    results.append(metrics)
    print(f'\n-> Joint F1={metrics.get("joint_f1", "?")}, Sent Acc={metrics.get("sent_acc", "?")}')

## 2. Comparison Table

In [ ]:
NO_RET_JOINT_F1 = 0.7696
NO_RET_SENT_ACC = 0.8987
AUX_JOINT_F1 = 0.7273

print('=' * 80)
print('COMPARISON TABLE — 4 New Experiments')
print('=' * 80)
print(f'{"Approach":<22s} | {"Cat F1":>7s} | {"Joint F1":>8s} | {"vs NoRet":>8s} | {"Sent Acc|CC":>11s} | {"Sent MacF1":>10s}')
print('-' * 80)

# Print baselines for reference
print(f'{"No-Retrieval (ref)":<22s} | {0.8564:>7.4f} | {NO_RET_JOINT_F1:>8.4f} | {"—":>8s} | {NO_RET_SENT_ACC:>11.4f} | {0.8022:>10.4f}')
print(f'{"Aux Loss (ref)":<22s} | {0.8564:>7.4f} | {AUX_JOINT_F1:>8.4f} | {AUX_JOINT_F1 - NO_RET_JOINT_F1:>+8.4f} | {0.8492:>11.4f} | {0.7456:>10.4f}')
print('-' * 80)

for r in results:
    jf1 = r.get('joint_f1', 0)
    delta = jf1 - NO_RET_JOINT_F1
    marker = ' ***' if jf1 > NO_RET_JOINT_F1 else ''
    print(f'{r["name"]:<22s} | {r.get("cat_f1", 0):>7.4f} | {jf1:>8.4f} | {delta:>+8.4f} | {r.get("sent_acc", 0):>11.4f} | {r.get("sent_macf1", 0):>10.4f}{marker}')

print('-' * 80)
print('*** = beats no-retrieval baseline')

# Summary
best = max(results, key=lambda x: x.get('joint_f1', 0)) if results else None
if best:
    print(f'\nBest: {best["name"]} — Joint F1={best.get("joint_f1", 0):.4f}')
    if best.get('joint_f1', 0) > NO_RET_JOINT_F1:
        print('RETRIEVAL WINS!')
    else:
        print(f'Still behind no-retrieval by {NO_RET_JOINT_F1 - best.get("joint_f1", 0):.4f}')

## 3. Individual Results

In [ ]:
for exp in EXPERIMENTS:
    log_path = f'logs/joint_eval_{exp["tag"]}.md'
    if os.path.exists(log_path):
        print(f'\n{"=" * 60}')
        print(f'{exp["name"]}')
        print(f'{"=" * 60}')
        with open(log_path) as f:
            print(f.read())
    else:
        print(f'\n{exp["name"]}: no results')

## 4. Save Outputs

In [ ]:
output_dir = '/kaggle/working/outputs_p5_nb3'
os.makedirs(output_dir, exist_ok=True)

if os.path.exists('logs'):
    shutil.copytree('logs', f'{output_dir}/logs', dirs_exist_ok=True)
    print('logs/ copied')

print(f'\nOutputs saved to {output_dir}')